In [ ]:
# Install required libraries
!pip install transformers soundfile speechbrain accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 864.1/864.1 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.1/754.1 kB 50.7 MB/s eta 0:00:00


In [ ]:
# Hugging Face login
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!pip install datasets==2.18.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 16.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.2.0 which is incompatible.


### Load and Prepare the Dataset

In [ ]:
from datasets import load_dataset, Audio

# Load Romanian subset of VoxPopuli
dataset = load_dataset("facebook/voxpopuli", "ro", split="train", trust_remote_code=True)

# Shuffle and take 1/4 subset
dataset = dataset.shuffle(seed=42)
dataset = dataset.select(range(len(dataset) // 4))

# Cast audio column to 16kHz
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [ ]:
len(dataset)

6064

### Analyze Audio Durations

In [ ]:
# Add duration field
def add_duration(example):
    example["duration"] = len(example["audio"]["array"]) / 16000
    return example

dataset = dataset.map(add_duration)

# Print duration stats
durations = dataset["duration"]
print(f"Average duration: {sum(durations)/len(durations):.2f}s")
print(f"Max duration: {max(durations):.2f}s")
print(f"Min duration: {min(durations):.2f}s")

Map:   0%|          | 0/6064 [00:00<?, ? examples/s]

Average duration: 11.48s
Max duration: 104.54s
Min duration: 0.52s


### Load Models and Define Speaker Embedding Function

In [ ]:
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech
from speechbrain.pretrained import EncoderClassifier
import torch

# Load processor and model
processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")

# Load speaker embedding model
spk_model = EncoderClassifier.from_hparams(source="speechbrain/spkrec-xvect-voxceleb", run_opts={"device":"cuda" if torch.cuda.is_available() else "cpu"})

# Define speaker embedding function
def extract_speaker_embedding(example):
    waveform = torch.tensor(example["audio"]["array"]).unsqueeze(0)
    embedding = spk_model.encode_batch(waveform).squeeze(0).detach()
    example["speaker_embeddings"] = embedding.cpu().numpy()
    return example

dataset = dataset.map(extract_speaker_embedding)

/usr/local/lib/python3.12/dist-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _speechbrain_save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _speechbrain_load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _save
DEBUG:speechbrain.utils.checkpoi

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/433 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/232 [00:00<?, ?B/s]

spm_char.model:   0%|          | 0.00/238k [00:00<?, ?B/s]

...

classifier.ckpt:   0%|          | 0.00/15.9M [00:00<?, ?B/s]

DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["classifier"] = /root/.cache/huggingface/hub/models--speechbrain--spkrec-xvect-voxceleb/snapshots/56895a2df401be4150a159f3a1c653f00051d477/classifier.ckpt
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached


label_encoder.txt: 0.00B [00:00, ?B/s]

DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["label_encoder"] = /root/.cache/huggingface/hub/models--speechbrain--spkrec-xvect-voxceleb/snapshots/56895a2df401be4150a159f3a1c653f00051d477/label_encoder.txt
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder
DEBUG:speechbrain.utils.parameter_transfer:Redirecting (loading from local path): embedding_model -> /root/.cache/huggingface/hub/models--speechbrain--spkrec-xvect-voxceleb/snapshots/56895a2df401be4150a159f3a1c653f00051d477/embedding_model.ckpt
DEBUG:speechbrain.utils.parameter_transfer:Redirecting (loading from local path): mean_var_norm_emb -> /root/.cache/huggingface/hub/models--speechbrain--spkrec-xvect-voxceleb/snapshots/56895a2df401be4150a159f3a1c653f00051d477/mean_var_norm_emb.ckpt
DEBUG:speechbrain.utils.parameter_transfer:Redirecting (loading from local path): classifier -> /root/.cache/huggingface/hub/models--speechb

Map:   0%|          | 0/6064 [00:00<?, ? examples/s]

### Normalize Text and Analyze Vocabulary

In [ ]:
# Extract unique characters
all_chars = set("".join(dataset["normalized_text"]))
tokenizer_chars = set(processor.tokenizer.get_vocab().keys())

# Define Romanian normalization
import re

def normalize_text(example):
    text = example["normalized_text"]
    text = text.lower()
    text = re.sub("ă", "a", text)
    text = re.sub("â", "a", text)
    text = re.sub("î", "i", text)
    text = re.sub("ș", "s", text)
    text = re.sub("ț", "t", text)
    example["normalized_text"] = text
    return example

dataset = dataset.map(normalize_text)

### Speaker Filtering

In [ ]:
from collections import defaultdict

# Count samples per speaker
speaker_counts = defaultdict(int)
for speaker_id in dataset["speaker_id"]:
    speaker_counts[speaker_id] += 1

# Filter by speaker count range
def speaker_filter(example):
    count = speaker_counts[example["speaker_id"]]
    return 120 <= count <= 350

dataset = dataset.filter(speaker_filter)

### Prepare Data for Model Input

In [ ]:
def prepare_dataset(example):
    # Tokenize text
    input_ids = processor.tokenizer(example["normalized_text"]).input_ids

    # Process audio
    audio = example["audio"]["array"]
    speech_inputs = processor.feature_extractor(audio, sampling_rate=16000, return_tensors="pt")
    example["input_ids"] = input_ids
    example["labels"] = speech_inputs["input_features"].squeeze(0)
    return example

# Filter long tokenized texts
def filter_long_texts(example):
    return len(processor.tokenizer(example["normalized_text"]).input_ids) <= 200

dataset = dataset.filter(filter_long_texts)
dataset = dataset.map(prepare_dataset)

# Split dataset
train_test = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = train_test["train"]
eval_dataset = train_test["test"]

### Custom Data Collator

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List
import torch

@dataclass
class CustomCollator:
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        input_ids = [torch.tensor(f["input_ids"]) for f in features]
        labels = [f["labels"] for f in features]
        speaker_embeddings = [torch.tensor(f["speaker_embeddings"]) for f in features]

        batch = self.processor.tokenizer.pad({"input_ids": input_ids}, return_tensors="pt", padding=True)
        labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)
        speaker_embeddings = torch.stack(speaker_embeddings)

        batch["labels"] = labels
        batch["speaker_embeddings"] = speaker_embeddings
        return batch

collator = CustomCollator(processor=processor)

### Training

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="speecht5_finetuned_voxpopuli_ro",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    warmup_steps=200,
    max_steps=1000,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    per_device_eval_batch_size=4,
    save_steps=500,
    eval_steps=100,
    logging_steps=50,
    load_best_model_at_end=True,
    greater_is_better=False,
    label_names=["labels"],
    push_to_hub=False,
    report_to=[],
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collator,
    tokenizer=processor.tokenizer,
)

In [ ]:
trainer.train()

### Inference and Vocoder

In [ ]:
from transformers import SpeechT5HifiGan

# Load vocoder
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan").to("cuda")

# Pick test example
example = eval_dataset[0]
speaker_embeddings = torch.tensor(example["speaker_embeddings"]).unsqueeze(0).to("cuda")

text = "salutare tuturor, acesta este un exemplu de sinteza a vocii in limba romana"
inputs = processor(text=text, return_tensors="pt").to("cuda")

# Generate speech
generated_speech = model.generate_speech(inputs["input_ids"], speaker_embeddings, vocoder=vocoder)

# Save or play
import soundfile as sf
sf.write("generated_speech.wav", generated_speech.cpu().numpy(), samplerate=16000)